# 05 — Optional LSTM and TimeGPT Models

This notebook shows how to extend the PoC with deep-learning and foundation-model approaches.

The goal is not to make the repository depend on heavy libraries by default. Instead, this notebook shows clear integration points while keeping the core PoC reliable.


In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

SRC = PROJECT_ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

pd.set_option("display.max_columns", 80)
plt.rcParams["figure.figsize"] = (11, 4)


In [ ]:
from clinic_forecast.data import generate_synthetic_healthcare_data

data_path = PROJECT_ROOT / "data" / "raw" / "clinic_usage.csv"
if data_path.exists():
    usage = pd.read_csv(data_path, parse_dates=["date"])
else:
    usage, _, _ = generate_synthetic_healthcare_data()
    usage["date"] = pd.to_datetime(usage["date"])


## LSTM framing

LSTMs need sequences. For clinic demand forecasting, a simple framing is:

- input: previous 28 days of visits and exogenous features,
- output: next day of visits,
- train one global model across all clinics.

This is usually not the first model to deploy. It is included here to show how deep-learning forecasting can be evaluated against stronger baselines.


In [ ]:
try:
    import torch
    import torch.nn as nn

    class ClinicDemandLSTM(nn.Module):
        """Minimal LSTM regressor for clinic demand sequences."""

        def __init__(self, n_features: int, hidden_size: int = 32) -> None:
            super().__init__()
            self.lstm = nn.LSTM(input_size=n_features, hidden_size=hidden_size, batch_first=True)
            self.head = nn.Linear(hidden_size, 1)

        def forward(self, x: torch.Tensor) -> torch.Tensor:
            output, _ = self.lstm(x)
            last_state = output[:, -1, :]
            return self.head(last_state).squeeze(-1)

    print("PyTorch is installed. LSTM class is ready.")
except ImportError:
    print("PyTorch is not installed. Run `poetry install --with optional` to enable this section.")


## TimeGPT integration

TimeGPT is useful to benchmark a foundation model against local statistical and ML models. In this PoC it is optional and requires an API key.


In [ ]:
try:
    from clinic_forecast.models.optional_timegpt import timegpt_forecast

    sample = usage[usage["clinic_id"].isin(["CLINIC_001", "CLINIC_002"])].copy()
    print("TimeGPT function imported successfully.")
    print("Call `timegpt_forecast(sample, horizon=28)` after setting NIXTLA_API_KEY.")
except ImportError as exc:
    print(exc)


## Evaluation rule

Deep-learning or foundation models should only be adopted if they improve the operational objective. In this project that means lower forecast error and better staffing recommendations under rolling-origin validation.
